# Prefect Deployment

### Introduction

So far we have seen how to get set up with prefect, and build our first flow and tasks.  However, we have not yet seen how to *schedule* this flow so that it is called say every hour.  To do this, we'll need to define and a assign a prefect deployment.  Ok, let's get going.

### Building a deployment

Ok, if you look at the current codebase, we have a flow that's defined in our `index.py` file, and our flow function is the `get_data` function.

To create a deployment, first make sure that your local prefect server is started.  Navigate to the prefect codebase and running the following.

```bash
prefect server start
```

And we can create our deployment file with the following.

```bash
prefect deployment build ./index.py:get_data -n get-restaurants -q test
```

Ok, so this is how to make sense of the above command:

* `prefect deployment build` is the standard command for building a deployment
* `./index.py:get_data` specifies the entrypoint flow function, with the file followed by the name of the function
* `-n get-restaurants` specifies the deployment name.
* `-q test` is the work pool for the deployment. Where a work pool directs a scheduled runs to an agent.

Running the above creates a new file in our codebase called `get_data-deployment.yaml`.  If you look through the new yaml file, you'll see some of our specifications, as well as other information that prefect filled in.

```yaml
###
### A complete description of a Prefect Deployment for flow 'get-data'
###
name: get-restaurants
description: null
version: 1f5a2d8c7f74279cc07fbc62a533b808
# The work queue that will handle this deployment's runs
work_queue_name: test
work_pool_name: null
tags: []
parameters: None
....
```

### Adding a schedule

Ok, so our CLI command filled in a lot of info for our deployment, but one key attribute that is still not filled in is a [schedule](https://docs.prefect.io/2.10.12/concepts/schedules/).  The schedule will indicating how often we would like this deployment to be run.  So let's update the `schedule` attribute in our deployment yaml file.  To do so change:

```yaml
schedule: null
```

To the following:

```yaml
schedule:
  interval: 60
```

This will specify to run the deployment every minute.

### Adding a parameter

We're almost ready to apply our deployment, but it turns out our flow will currently break.  If you look at our flow, you'll see that it takes an argument of url.

```python
@flow
def get_data(url: str):
    receipts = find_receipts(url)
    return receipts
```

So we'll want our flow function `get_data` to get called with an argument for the url.  We can make sure it is by updating the deployment file's `parameters` attribute.

```yaml
parameters: {'url': 'HONDURAS MAYA CAFE & BAR LLC'}
```

This will invoke our flow function with a url value of the Honduras string.

### Applying the deployment

Ok, now that we've updated the schedule and the parameters, it's time to apply the deployment.  Let's do so with by putting the following CLI command in the terminal:

```bash
prefect deployment apply get_data-deployment.yaml
```

Now that we applied the deployment, you can see information about the deployment if you go to prefect cloud, click on deployments, and then click on the name of the deployment.

<img src="./see-deployment.png" width="50%">

But unfortunately, this deployment is unhealthy. 

<img src="./unhealthy.png">

The issue is that we need to assign a work pool that will handle this deployment.

<img src="./workqueue.png">

Ok, so now our deployment is working.  And it's running our flow every minute.

Finally, if we would like to stop a deployment, we can do so by going to prefect cloud, clicking on deployments and clicking on the button over to the right so that it is no longer green.

<img src="./deployment-pause.png">

### Summary

In this lesson we saw how to schedule a flow with a deployment.  We did so by first creating our deployment yaml file.

```bash
prefect deployment build ./index.py:get_data -n get-restaurants -q test
```

And then we updated our deployment file's schedule and parameters values.

```yaml
schedule:
  interval: 60
```

```yaml
parameters: {'url': 'HONDURAS MAYA CAFE & BAR LLC'}
```

With this our flow will be run every 60 seconds, and the flow will be run with the url parameter passed through as an argument.

We then applied the deployment.

```bash
prefect deployment apply get_data-deployment.yaml
```

And finally assigned a work pool for the deployment.

```bash
prefect agent start --pool default-agent-pool --work-queue test
```